In [16]:
from time import time
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error

In [17]:
import inspect

def instantiate_model(model_class, random_state=42):
    if 'random_state' in inspect.signature(model_class.__init__).parameters:
        model = model_class(random_state=random_state)
    else:
        model = model_class()
    return model

In [18]:
df_train = pd.read_csv('regression_dataset_splits/app_train_split.csv')
df_test = pd.read_csv('regression_dataset_splits/app_test_split.csv')
df_val = pd.read_csv('regression_dataset_splits/app_val_split.csv')

frames = [df_train, df_val, df_test]

df = pd.concat(frames)

df["target"] = df["P2O3ratio"]

df

/tmp/ipykernel_71746/2476601037.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["target"] = df["P2O3ratio"]


,UID,Chemical,Time,Na,Cu,Fe,Mn,Ni,P2O3ratio,formula,...,var_gordy_en_range,var_gordy_en_mode,var_mb_en_sum,var_mb_en_avg,var_mb_en_dev,var_mb_en_min,var_mb_en_max,var_mb_en_range,var_mb_en_mode,target
0,U00006,NaCu0.5Mn0.5O2,12,1.00,0.500000,0.000000,0.500000,0.000000,1.00,NaCu0.5Mn0.5O2,...,0.027073,0.271435,0.074112,0.074112,0.029336,0.044776,0.103448,0.058672,0.103448,1.00
1,U00008,NaFe0.5Mn0.5O2,12,1.00,0.000000,0.500000,0.500000,0.000000,1.00,NaFe0.5Mn0.5O2,...,0.011477,0.266255,0.057325,0.057325,0.006369,0.050955,0.063694,0.012739,0.063694,1.00
2,U00010,NaMn0.5Ni0.5O2,12,1.00,0.000000,0.000000,0.500000,0.500000,1.00,NaMn0.5Ni0.5O2,...,0.024911,0.273596,0.070581,0.070581,0.025805,0.044776,0.096386,0.051609,0.096386,1.00
3,U00013,NaCu1/3Mn1/3Ni1/3O2,12,1.00,0.333333,0.000000,0.333333,0.333333,1.00,NaCu1/3Mn1/3Ni1/3O2,...,0.035437,0.271435,0.080859,0.080940,0.025826,0.044776,0.103448,0.058672,0.103448,1.00
4,U00014,NaFe1/3Mn1/3Ni1/3O2,12,1.00,0.000000,0.333333,0.333333,0.333333,0.00,NaFe1/3Mn1/3Ni1/3O2,...,0.032253,0.266255,0.068217,0.068285,0.021318,0.044776,0.096386,0.051609,0.063694,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21,U00324,Na0.75Ni2/3Cu1/6Mn1/6O2,3,0.75,0.166667,0.000000,0.166667,0.666667,0.00,Na0.75Ni2/3Cu1/6Mn1/6O2,...,0.094831,0.306872,0.088393,0.088305,0.019717,0.044776,0.103448,0.058672,0.094595,0.00
22,U00332,Na0.75Ni2/3Fe1/9Cu1/9Mn1/9O2,3,0.75,0.111111,0.111111,0.111111,0.666667,0.73,Na0.75Ni2/3Fe1/9Cu1/9Mn1/9O2,...,0.100012,0.306872,0.087162,0.087162,0.018334,0.044776,0.103448,0.058672,0.094595,0.73
23,U00335,Na0.75Ni1/9Cu8/27Fe8/27Mn8/27O2,3,0.75,0.296296,0.296296,0.296296,0.111111,0.33,Na0.75Ni1/9Cu8/27Fe8/27Mn8/27O2,...,0.100012,0.271435,0.071410,0.071482,0.028383,0.000000,0.103448,0.103448,0.103448,0.33
24,U00338,Na0.75Fe7/18Ni7/18Cu1/9Mn1/9O2,3,0.75,0.111111,0.388889,0.111111,0.388889,0.19,Na0.75Fe7/18Ni7/18Cu1/9Mn1/9O2,...,0.100012,0.266255,0.078572,0.078572,0.020002,0.044776,0.103448,0.058672,0.063694,0.19


In [19]:
# Define which columns are your features (drop non-numeric / target cols)
feature_cols = [col for col in df_train.columns if col not in ['UID','Chemical','P2O3ratio','target', 'formula', 'Cat_Success']]

# Training
X_full = df[feature_cols].values
y_full = df['target'].values

In [20]:
groups = df['Chemical'].values
print(f'Unique compositions: {len(np.unique(groups))}')
print(pd.Series(groups).value_counts().head(10))  # check if any Chemical appears more than once

Unique compositions: 99
NaCu0.5Mn0.5O2                  3
NaFe0.5Mn0.5O2                  3
NaMn0.5Ni0.5O2                  3
NaCu1/3Mn1/3Ni1/3O2             3
NaFe1/3Mn1/3Ni1/3O2             3
NaCu0.25Fe0.25Mn0.25Ni0.25O2    3
Na0.75MnO2                      3
Na0.75Cu0.5Mn0.5O2              3
Na0.75Mn0.5Ni0.5O2              3
NaMn2/3Cu1/6Fe1/6O2             3
Name: count, dtype: int64


In [21]:
from sklearn.model_selection import GroupKFold

groups = df['Chemical'].values  # group by exact composition

gkf = GroupKFold(n_splits=5)
results = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X_full, y_full, groups=groups)):
    X_tr, X_te = X_full[train_idx], X_full[test_idx]
    y_tr, y_te = y_full[train_idx], y_full[test_idx]

    fold_model = instantiate_model(RandomForestRegressor, random_state=42)
    fold_model.fit(X_tr, y_tr)

    y_pred = fold_model.predict(X_te)
    results.append({
        'fold': fold + 1,
        'n_train': len(y_tr),
        'n_test': len(y_te),
        'r2': r2_score(y_te, y_pred),
        'mae': mean_absolute_error(y_te, y_pred),
        'rmse': root_mean_squared_error(y_te, y_pred),
    })

df_cv = pd.DataFrame(results)
print(df_cv.to_string(index=False))
print(f'\nMean R²:   {df_cv["r2"].mean():0.4f} ± {df_cv["r2"].std():0.4f}')
print(f'Mean MAE:  {df_cv["mae"].mean():0.4f} ± {df_cv["mae"].std():0.4f}')
print(f'Mean RMSE: {df_cv["rmse"].mean():0.4f} ± {df_cv["rmse"].std():0.4f}')

 fold  n_train  n_test       r2      mae     rmse
    1      210      53 0.405389 0.178626 0.250917
    2      210      53 0.009914 0.165709 0.278404
    3      210      53 0.695915 0.133202 0.216642
    4      211      52 0.708488 0.146617 0.206593
    5      211      52 0.575181 0.153787 0.242011

Mean R²:   0.4790 ± 0.2891
Mean MAE:  0.1556 ± 0.0175
Mean RMSE: 0.2389 ± 0.0285


In [23]:
for fold, (train_idx, test_idx) in enumerate(gkf.split(X_full, y_full, groups=groups)):
    test_compositions = df.iloc[test_idx][['Chemical', 'P2O3ratio']]
    print(f'\n--- Fold {fold + 1} (n_test={len(test_idx)}, R²={results[fold]["r2"]:.4f}) ---')
    print(test_compositions.to_string(index=False))


--- Fold 1 (n_test=53, R²=0.4054) ---
                        Chemical  P2O3ratio
                  NaMn0.5Ni0.5O2       1.00
         Na0.75Fe0.3Mn0.3Ni0.3O2       0.96
Na0.75Cu0.25Fe0.25Mn0.25Ni0.25O2       0.98
                  NaMn0.5Ni0.5O2       1.00
Na0.75Cu0.25Fe0.25Mn0.25Ni0.25O2       0.57
                  NaMn0.5Ni0.5O2       1.00
        NaCu2/3Ni1/9Mn1/9Fe1/9O2       1.00
        NaNi2/3Fe1/9Cu1/9Mn1/9O2       1.00
      NaMn7/18Ni7/18Fe1/9Cu1/9O2       1.00
         Na0.75Mn2/3Fe1/6Ni1/6O2       1.00
         Na0.75Cu2/3Fe1/6Mn1/6O2       1.00
          Na0.5Mn2/3Cu1/6Fe1/6O2       1.00
          Na0.5Cu2/3Ni1/6Mn1/6O2       1.00
        NaCu2/3Ni1/9Mn1/9Fe1/9O2       1.00
        NaNi2/3Fe1/9Cu1/9Mn1/9O2       0.26
      NaMn7/18Ni7/18Fe1/9Cu1/9O2       1.00
         Na0.75Mn2/3Fe1/6Ni1/6O2       1.00
         Na0.75Fe2/3Ni1/6Mn1/6O2       0.05
         Na0.75Cu2/3Fe1/6Mn1/6O2       0.88
          Na0.5Mn2/3Cu1/6Fe1/6O2       1.00
          Na0.5Cu2/3Ni1/6Mn1/6O2     